# OpenCV: распознавание монет на реальной фотографии

Этот ноутбук заменяет старый пример с `coins.webp`.
Он использует реальный файл `verified_materials/assets/coins_real.jpg`
и находит монеты методом `HoughCircles`.

Важно: прямоугольный сувенир/пластина на фото не считается монетой.


In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

cwd = Path.cwd()
MATERIALS = cwd / "verified_materials" if (cwd / "verified_materials").exists() else cwd
ASSETS = MATERIALS / "assets"
OUTPUTS = MATERIALS / "outputs"
OUTPUTS.mkdir(parents=True, exist_ok=True)

image_path = ASSETS / "coins_real.jpg"
img = cv2.imread(str(image_path))
if img is None:
    raise FileNotFoundError(image_path)

print("OpenCV:", cv2.__version__)
print("Image:", image_path)
print("Shape:", img.shape)


In [ ]:
rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(7, 9))
plt.imshow(rgb)
plt.axis("off")
plt.title("Исходная фотография")
plt.show()

cv2.imwrite(str(OUTPUTS / "opencv_real_coins_input.jpg"), img)


In [ ]:
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
gray_blur = cv2.medianBlur(gray, 7)
edges = cv2.Canny(gray_blur, 30, 90)

cv2.imwrite(str(OUTPUTS / "opencv_real_coins_edges.png"), edges)

plt.figure(figsize=(7, 9))
plt.imshow(edges, cmap="gray")
plt.axis("off")
plt.title("Границы Canny")
plt.show()


In [ ]:
circles = cv2.HoughCircles(
    gray_blur,
    cv2.HOUGH_GRADIENT,
    dp=1.2,
    minDist=130,
    param1=90,
    param2=40,
    minRadius=55,
    maxRadius=140,
)

result = img.copy()
coins_count = 0

if circles is not None:
    circles = np.uint16(np.around(circles))
    # Sort for stable numbering: top-to-bottom, left-to-right.
    sorted_circles = sorted(circles[0, :], key=lambda c: (int(c[1]), int(c[0])))
    for i, (x, y, r) in enumerate(sorted_circles, start=1):
        coins_count += 1
        cv2.circle(result, (int(x), int(y)), int(r), (0, 190, 0), 4)
        cv2.circle(result, (int(x), int(y)), 3, (0, 0, 255), -1)
        cv2.putText(
            result,
            f"#{i}",
            (int(x) - 20, int(y) - int(r) - 12),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.9,
            (0, 80, 0),
            3,
            cv2.LINE_AA,
        )

out_path = OUTPUTS / "opencv_real_coins_detected.jpg"
cv2.imwrite(str(out_path), result)

print(f"Detected coins: {coins_count}")
print("Saved:", out_path)

plt.figure(figsize=(7, 9))
plt.imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title(f"Найдено монет: {coins_count}")
plt.show()
